# 01 - PDF Extraction and Text Cleaning

This notebook explains and demonstrates the first stage of the Manufacturing Maintenance RAG Assistant.

## Learning objectives

By the end, you should be able to explain:

- Why a RAG system needs machine-readable text instead of visual PDF pages
- Why source document and page metadata must be preserved
- How repeated headers, footers, page numbers, and broken line wrapping affect retrieval
- Why direct extraction is preferable to OCR for digitally generated PDFs
- Why the processed corpus is stored as JSON Lines (JSONL)


## Implementation map - where extraction happens

The reusable implementation is in `src/extract_text.py`. Open it beside this notebook and locate: `find_repeated_edge_lines()`, `clean_page_text()`, `extract_document()`, and `extract_corpus()`.

- `find_repeated_edge_lines()` learns recurring headers and footers from page edges.
- `clean_page_text()` contains the main normalization and removal rules.
- `extract_document()` reads one PDF and creates page records with raw and cleaned text.
- `extract_corpus()` processes all PDFs and writes `pages.jsonl` plus diagnostics.

These function names are the exact implementation checkpoints you should understand.

## Pipeline

```text
NIST PDF -> page-level extraction -> cleaning -> metadata -> pages.jsonl
```

We keep both `raw_text` and `cleaned_text`. Raw text supports debugging and traceability; cleaned text is used by later chunking and embedding steps.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
from pypdf import PdfReader
try:
    from IPython.display import display
except ImportError:
    display = print

# Support execution from either the project root or the notebooks folder.
current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name == 'notebooks' else current_dir
sys.path.insert(0, str(project_root))

from src.extract_text import (
    clean_page_text,
    extract_corpus,
    find_repeated_edge_lines,
)

raw_dir = project_root / 'data' / 'raw'
processed_dir = project_root / 'data' / 'processed'
pdf_paths = sorted(raw_dir.glob('*.pdf'))

print(f'Project root: {project_root}')
print(f'PDF documents found: {len(pdf_paths)}')
for path in pdf_paths:
    print('-', path.name)

Project root: <USER_HOME>\Documents\ChatGPT\Data Analytics\engineering-document-rag-assistant
PDF documents found: 3
- NIST.AMS.100-18.pdf
- NIST.AMS.100-2.pdf
- NIST.AMS.100-34.pdf


## Implementation checkpoint - page extraction and metadata

**Open in `src/extract_text.py`:** `extract_document()`.

**Input:** one PDF path. **Output:** page dictionaries and a diagnostic report.

**Key implementation:** `PdfReader` extracts each page; `enumerate(..., start=1)` preserves human-readable PDF page numbers; both `raw_text` and `cleaned_text` are retained.

**You should be able to explain:** citation metadata must be captured during ingestion. It cannot be reconstructed reliably after text from multiple pages has been mixed.

## Inspect direct extraction

`pypdf` reads the embedded text layer. This is faster and generally more accurate than OCR when a PDF was digitally generated. We begin with one document and inspect its page-level extraction.

In [2]:
sample_pdf = raw_dir / 'NIST.AMS.100-34.pdf'
reader = PdfReader(sample_pdf)
raw_pages = [(page.extract_text() or '') for page in reader.pages]

page_statistics = pd.DataFrame({
    'pdf_page': range(1, len(raw_pages) + 1),
    'raw_characters': [len(text.strip()) for text in raw_pages],
    'replacement_characters': [text.count('\ufffd') for text in raw_pages],
})

display(page_statistics.describe())
display(page_statistics.head())

,pdf_page,raw_characters,replacement_characters
count,52.000000,52.000000,52.0
mean,26.500000,2237.250000,0.0
std,15.154757,1016.791105,0.0
min,1.000000,278.000000,0.0
25%,13.750000,1795.500000,0.0
50%,26.500000,2271.500000,0.0
75%,39.250000,2973.250000,0.0
max,52.000000,4012.000000,0.0


,pdf_page,raw_characters,replacement_characters
0,1,283,0
1,2,583,0
2,3,668,0
3,4,2579,0
4,5,4012,0


## Detect repeated headers and footers

The extraction module examines the first and last five non-empty lines of each page. A short edge line appearing on at least 30% of populated pages is treated as repeated page furniture. This avoids manually hard-coding every document header.

In [3]:
repeated_lines = find_repeated_edge_lines(raw_pages)
print('Repeated edge lines detected:')
for line in sorted(repeated_lines):
    print('-', line)

Repeated edge lines detected:
- This publication is available free of charge from: https://doi.org/10.6028/NIST.AMS.100-34


## Compare raw and cleaned text

Cleaning removes repeated page furniture, isolated page numbers, and unnecessary whitespace. It also repairs words split across line boundaries, such as `manufac-\nturing`.

In [4]:
sample_page_number = 16
sample_raw_text = raw_pages[sample_page_number - 1]
sample_cleaned_text = clean_page_text(sample_raw_text, repeated_lines)

print('RAW TEXT (first 1,200 characters)')
print('-' * 80)
print(sample_raw_text[:1200])
print('\nCLEANED TEXT (first 1,200 characters)')
print('-' * 80)
print(sample_cleaned_text[:1200])

RAW TEXT (first 1,200 characters)
--------------------------------------------------------------------------------
 
 
10 
This publication is available free of charge from: https://doi.org/10.6028/NIST.AMS.100-34 
 
necessary PM, this excess can lead to wasted money in unnecessary labor hours and 
materials. PM is planned downtime which is still downtime. The more time a piece of 
equipment is down, the less opportunity there is for it to be operating to produce a product 
or perform a process to support the organization’s revenue stream.  
 
An organization typically designs the PM strategy to be a tradeoff between the potential 
for excess maintenance and the risk of not doing enoug h maintenance thereby increasing 
the presence of a fault and failure lead ing to RM. The potential for excess maintenance 
leading to excess cost is also balanced with the invested cost of a Predictive Maintenance 
strategy, discussed in detail in section 2.3. 
2.3. Predictive Maintenance 
Predictive Ma

## Extract the complete corpus

Each retained page becomes one JSON object containing its source document, PDF page number, raw text, cleaned text, and character count. Pages with fewer than 30 useful characters are skipped and recorded in the diagnostic report.

In [5]:
pages_file = processed_dir / 'pages.jsonl'
report_file = processed_dir / 'extraction_report.json'

extract_corpus(raw_dir, pages_file, report_file)

Processed 3 documents
Retained 159 pages
Wrote page corpus to <USER_HOME>\Documents\ChatGPT\Data Analytics\engineering-document-rag-assistant\data\processed\pages.jsonl
Wrote extraction report to <USER_HOME>\Documents\ChatGPT\Data Analytics\engineering-document-rag-assistant\data\processed\extraction_report.json


## Inspect the processed page records

JSONL stores one independent JSON object per line. This makes the corpus easy to inspect, stream, and load into Pandas or Hugging Face Datasets.

In [6]:
records = [
    json.loads(line)
    for line in pages_file.read_text(encoding='utf-8').splitlines()
]
pages_df = pd.DataFrame(records)

print(f'Retained page records: {len(pages_df)}')
display(
    pages_df.groupby('document')
    .agg(
        retained_pages=('pdf_page', 'count'),
        median_cleaned_characters=('cleaned_character_count', 'median'),
        minimum_cleaned_characters=('cleaned_character_count', 'min'),
    )
)
display(pages_df[['document', 'pdf_page', 'cleaned_character_count']].head())

Retained page records: 159


,retained_pages,median_cleaned_characters,minimum_cleaned_characters
document,,,
NIST.AMS.100-18,43,2269.0,137
NIST.AMS.100-2,64,2783.0,384
NIST.AMS.100-34,52,2125.5,179


,document,pdf_page,cleaned_character_count
0,NIST.AMS.100-18,1,137
1,NIST.AMS.100-18,2,391
2,NIST.AMS.100-18,3,737
3,NIST.AMS.100-18,5,2269
4,NIST.AMS.100-18,6,1937


In [7]:
report = json.loads(report_file.read_text(encoding='utf-8'))
report_rows = [
    {
        'document': item['document'],
        'total_pdf_pages': item['total_pdf_pages'],
        'retained_pages': item['retained_pages'],
        'skipped_pages': item['skipped_pages'],
        'replacement_characters': item['replacement_characters_in_raw_text'],
    }
    for item in report['documents']
]
display(pd.DataFrame(report_rows))

,document,total_pdf_pages,retained_pages,skipped_pages,replacement_characters
0,NIST.AMS.100-18,45,43,"[4, 38]",0
1,NIST.AMS.100-2,66,64,"[2, 4]",0
2,NIST.AMS.100-34,52,52,[],0


## Where the extracted text is stored

The source PDFs remain unchanged in `data/raw/`. The extraction pipeline writes two files into `data/processed/`:

- `pages.jsonl`: one page record per line, including raw text, cleaned text, document name, and PDF page number
- `extraction_report.json`: document-level diagnostics, including retained pages, skipped pages, and repeated lines removed

The notebook can safely rerun extraction because these files are reproducible outputs of the source PDFs and cleaning code.

In [8]:
print('Page corpus:', pages_file.resolve())
print('Exists:', pages_file.exists(), '| Size:', f'{pages_file.stat().st_size / 1024:.1f} KB')
print('Extraction report:', report_file.resolve())
print('Exists:', report_file.exists(), '| Size:', f'{report_file.stat().st_size / 1024:.1f} KB')

example_record = records[10]
print('\nExample stored record (text shortened for display):')
print({
    'document': example_record['document'],
    'filename': example_record['filename'],
    'pdf_page': example_record['pdf_page'],
    'raw_text_preview': example_record['raw_text'][:180],
    'cleaned_text_preview': example_record['cleaned_text'][:180],
    'cleaned_character_count': example_record['cleaned_character_count'],
})

Page corpus: <USER_HOME>\Documents\ChatGPT\Data Analytics\engineering-document-rag-assistant\data\processed\pages.jsonl
Exists: True | Size: 794.0 KB
Extraction report: <USER_HOME>\Documents\ChatGPT\Data Analytics\engineering-document-rag-assistant\data\processed\extraction_report.json
Exists: True | Size: 1.5 KB

Example stored record (text shortened for display):
{'document': 'NIST.AMS.100-18', 'filename': 'NIST.AMS.100-18.pdf', 'pdf_page': 12, 'raw_text_preview': 'This publication is available free of charge from: https://doi.org/10.6028/NIST.AMS.100-18 \nmaintenance staff with spare parts being another 32 %, as seen in Figure 2.1. Komonen es', 'cleaned_text_preview': 'maintenance staff with spare parts being another 32 %, as seen in Figure 2.1. Komonen estimates that industrial maintenance is 5.5 % of company turnover (i.e., sales); however, it ', 'cleaned_character_count': 2443}


## Cleaning process, step by step

For each document, the pipeline performs the following operations:

1. Open the PDF with `PdfReader`.
2. Extract every page's embedded text. If extraction returns `None`, use an empty string.
3. Inspect the first and last five non-empty lines of every populated page.
4. Count repeated edge lines. A line found on at least 30% of pages is treated as a header or footer.
5. Process each page line by line.
6. Normalize repeated whitespace to one space.
7. Remove detected headers, DOI running lines, empty lines, and isolated page numbers.
8. Repair words hyphenated across a line break, such as `manufac-\nturing`.
9. Join ordinary wrapped lines while retaining useful structural line breaks.
10. Skip a page if fewer than 30 useful characters remain.
11. Save both raw and cleaned text with source-page metadata.

The raw version is deliberately retained. If a cleaning rule damages text, we can compare the two versions and rebuild the corpus without returning to OCR.

In [9]:
import inspect

print('The cleaning function used by the notebook and reusable pipeline:')
print(inspect.getsource(clean_page_text))

The cleaning function used by the notebook and reusable pipeline:
def clean_page_text(raw_text: str, repeated_lines: set[str]) -> str:
    """Remove page furniture and normalize PDF line wrapping."""
    retained_lines: list[str] = []

    for raw_line in raw_text.splitlines():
        line = normalize_line(raw_line)
        if not line or line in repeated_lines:
            continue

        # This running header is common in the selected NIST publications.
        if line.lower().startswith("this publication is available free of charge from:"):
            continue

        # Remove isolated PDF page numbers while retaining numbers in prose.
        if re.fullmatch(r"\d{1,3}", line):
            continue

        retained_lines.append(line)

    text = "\n".join(retained_lines)

    # Repair words split across PDF line boundaries, e.g. "manufac-\nturing".
    text = re.sub(r"(?<=[A-Za-z])[-‐]\n(?=[a-z])", "", text)

    # Join ordinary wrapped lines while retaining likely headings an

## Manual quality inspection

Automated checks catch missing data and repeated headers, but they cannot determine whether prose, lists, or tables remain readable. We therefore inspect a fixed random sample and one known page. A fixed `random_state` makes the sample reproducible.

In [10]:
pd.set_option('display.max_colwidth', 1000)

quality_sample = pages_df.sample(5, random_state=42)
for _, row in quality_sample.iterrows():
    print('=' * 100)
    print(f"Document: {row['document']} | PDF page: {row['pdf_page']}")
    print('-' * 100)
    print(row['cleaned_text'][:1500])
    print()

known_example = pages_df[
    (pages_df['document'] == 'NIST.AMS.100-34')
    & (pages_df['pdf_page'] == 16)
].iloc[0]

print('=' * 100)
print('Known extraction-artifact example: NIST.AMS.100-34, PDF page 16')
print('-' * 100)
print(known_example['cleaned_text'][:1800])

Document: NIST.AMS.100-2 | PDF page: 38
----------------------------------------------------------------------------------------------------
Barriers: Current status of the PHM framework in various industries is unclear; a PHM framework that includes standards is lacking, leading to inconsistencies in performance assessment.
Approach: The development of the framework would involve: 1) performing a benchmark for current state, 2) cataloging
Key Performance Indicators (KPIs), and 3) defining PHM standard architecture and traceability. A standard framework will provide consistency in assessment of PHM performance and support new standards development.
FIGURE 3-1. OVERARCHING ARCHITECTURE FRAMEWORK FOR PHM
WITH STANDARDS AND KPIS
Roadmap Action Plan
Targets/ Capabilities
Milestones
1-2 years w Benchmark the current state of machine monitoring in select industries, such as oil and gas, automotive, aerospace, and pharma/medical w Identification of two key PHM scenarios to benchmark from each

## Why NLTK is not required here

NLTK is useful for many traditional NLP workflows, but it is not required by RAG. For this project we will use lightweight sentence-boundary rules plus the Hugging Face tokenizer used by the embedding model. This keeps chunk length aligned with the transformer's actual token limits.

We will **not** remove stop words or stem the corpus before embedding. Transformer embedding models use word order and surrounding context; changing `the bearing is not damaged` into something like `bear damag` can remove negation and reduce meaning. We therefore preserve natural text.

## Chunking-readiness audit

Before chunking, we verify structural integrity and quantify remaining extraction artifacts. A corpus is ready for initial chunking when records are complete, source-page metadata is unique, text is non-empty, decoding is clean, and repeated page furniture has been removed. Minor isolated spacing artifacts can be documented and revisited if retrieval tests show an effect.

In [11]:
from collections import Counter
import re

required_fields = {
    'document', 'filename', 'pdf_page', 'raw_text',
    'cleaned_text', 'cleaned_character_count'
}
page_keys = list(zip(pages_df['document'], pages_df['pdf_page']))

readiness = {
    'records': len(pages_df),
    'documents': pages_df['document'].nunique(),
    'records_missing_fields': sum(bool(required_fields - set(record)) for record in records),
    'empty_cleaned_text': int(pages_df['cleaned_text'].str.strip().eq('').sum()),
    'duplicate_document_pages': sum(count > 1 for count in Counter(page_keys).values()),
    'character_count_mismatches': sum(
        len(record['cleaned_text']) != record['cleaned_character_count']
        for record in records
    ),
    'replacement_characters': int(pages_df['cleaned_text'].str.count('\ufffd').sum()),
    'leftover_running_headers': int(
        pages_df['cleaned_text'].str.contains(
            'This publication is available free of charge from:',
            regex=False,
        ).sum()
    ),
    'pages_with_known_split_words': int(
        pages_df['cleaned_text'].str.contains(r'\b(?:enoug h|lead ing)\b', regex=True).sum()
    ),
    'pages_with_space_before_hyphen': int(
        pages_df['cleaned_text'].str.contains(r'\b[A-Za-z]{3,}\s+-[A-Za-z]{2,}\b', regex=True).sum()
    ),
}

display(pd.DataFrame([readiness]).T.rename(columns={0: 'value'}))

blocking_failures = [
    name for name in (
        'records_missing_fields', 'empty_cleaned_text',
        'duplicate_document_pages', 'character_count_mismatches',
        'replacement_characters', 'leftover_running_headers',
    )
    if readiness[name] != 0
]

print('Decision:', 'NOT READY' if blocking_failures else 'READY FOR INITIAL CHUNKING')
print('Blocking failures:', blocking_failures or 'None')
print('Non-blocking limitation: isolated spacing artifacts remain and will be monitored during retrieval evaluation.')

,value
records,159
documents,3
records_missing_fields,0
empty_cleaned_text,0
duplicate_document_pages,0
character_count_mismatches,0
replacement_characters,0
leftover_running_headers,0
pages_with_known_split_words,1
pages_with_space_before_hyphen,9


Decision: READY FOR INITIAL CHUNKING
Blocking failures: None
Non-blocking limitation: isolated spacing artifacts remain and will be monitored during retrieval evaluation.


## Validation checks

Assertions make assumptions executable. If an assumption stops being true after a code or corpus change, the notebook raises an error instead of silently producing poor data.

In [12]:
assert len(pages_df) == 159, 'Unexpected retained-page count'
assert pages_df['document'].nunique() == 3, 'Expected three source documents'
assert (pages_df['cleaned_character_count'] >= 30).all()
assert pages_df['cleaned_text'].str.contains('This publication is available free of charge from:', regex=False).sum() == 0
assert pages_df['raw_text'].str.count('\ufffd').sum() == 0

print('All extraction validation checks passed.')

All extraction validation checks passed.


## What you should be able to explain

> I extracted the corpus at page level using pypdf, preserved source-page metadata for citations, retained raw text for traceability, removed repeated page furniture, repaired common line-wrapping artifacts, skipped visually blank pages, and stored the processed corpus as JSONL for downstream chunking.

## Before moving to chunking

Inspect several records containing normal prose, headings, lists, and tables. Write down any cleaning problems you observe. The next notebook will explain chunk size, overlap, metadata propagation, and retrieval trade-offs.

In [13]:
pd.set_option("display.max_colwidth", 1000)

sample = pages_df.sample(5, random_state=42)

for _, row in sample.iterrows():
    print("=" * 100)
    print(f"Document: {row['document']} | PDF page: {row['pdf_page']}")
    print("-" * 100)
    print(row["cleaned_text"][:2000])
    print()

Document: NIST.AMS.100-2 | PDF page: 38
----------------------------------------------------------------------------------------------------
Barriers: Current status of the PHM framework in various industries is unclear; a PHM framework that includes standards is lacking, leading to inconsistencies in performance assessment.
Approach: The development of the framework would involve: 1) performing a benchmark for current state, 2) cataloging
Key Performance Indicators (KPIs), and 3) defining PHM standard architecture and traceability. A standard framework will provide consistency in assessment of PHM performance and support new standards development.
FIGURE 3-1. OVERARCHING ARCHITECTURE FRAMEWORK FOR PHM
WITH STANDARDS AND KPIS
Roadmap Action Plan
Targets/ Capabilities
Milestones
1-2 years w Benchmark the current state of machine monitoring in select industries, such as oil and gas, automotive, aerospace, and pharma/medical w Identification of two key PHM scenarios to benchmark from each

In [14]:
record = pages_df[
    (pages_df["document"] == "NIST.AMS.100-34")
    & (pages_df["pdf_page"] == 16)
].iloc[0]

print(record["cleaned_text"])

necessary PM, this excess can lead to wasted money in unnecessary labor hours and materials. PM is planned downtime which is still downtime. The more time a piece of equipment is down, the less opportunity there is for it to be operating to produce a product or perform a process to support the organization’s revenue stream.
An organization typically designs the PM strategy to be a tradeoff between the potential for excess maintenance and the risk of not doing enoug h maintenance thereby increasing the presence of a fault and failure lead ing to RM. The potential for excess maintenance leading to excess cost is also balanced with the invested cost of a Predictive Maintenance strategy, discussed in detail in section 2.3.
2.3. Predictive Maintenance
Predictive Maintenance (PdM) is a strategy that dictates maintenance activities based upon measures of reliability and/or condition. Reliability -centered maintenance (RCM) and condition-based maintenance (CBM) are under the PdM strategy umbre